# Northbridge Policy Briefing Assistant  
**Department:** Advocacy & Community Issues  
**Course:** GBA 479 – Final Project  

This prototype demonstrates a retrieval-grounded policy assistant that generates evidence-based responses using internal Northbridge documentation, while enforcing guardrails to prevent unsupported claims or out-of-scope positions.

In [13]:
!pip -q install pypdf scikit-learn openai

In [14]:
# ==============================================================
# NORTHBRIDGE POLICY BRIEFING ASSISTANT
# Advocacy & Community Issues Department
# GBA 479 – Final Project
# ==============================================================


# ==============================================================
# IMPORTS & OPTIONAL LLM INITIALIZATION
# ==============================================================

import os
import json
import re

from dataclasses import dataclass
from typing import List, Dict, Any

from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Optional OpenAI integration.
# The assistant also works without an API key using deterministic logic.

USE_OPENAI = bool(os.getenv("OPENAI_API_KEY"))
client = None

if USE_OPENAI:
    from openai import OpenAI
    client = OpenAI()


# ==============================================================
# DATA STRUCTURE: DOCUMENT CHUNK
# ==============================================================

@dataclass
class Chunk:
    doc_name: str
    page_num: int
    text: str


# ==============================================================
# APPROVED NORTHBRIDGE DOCUMENT SOURCES
# ==============================================================

PDFS = [
    (
        "Northbridge_Strategic_Plan.pdf",
        "Northbridge Strategic Plan (2025–2028)"
    ),
    (
        "Northbridge_Policy_Brief_Food.pdf",
        "Policy Brief: Food Access & Equity in Eastview (Mar 2025)"
    ),
    (
        "Northbridge_Policy_Brief_Volunteers.pdf",
        "Policy Brief: Strengthening Volunteer Capacity (Apr 2025)"
    ),
    (
        "Northbridge_Evaluation.pdf",
        "Program Evaluation Summary (2025)"
    ),
    (
        "Northbridge_Advocacy_FAQ.pdf",
        "Advocacy & Community Issues FAQ (2026)"
    ),
    (
        "Northbridge_Content_Standards.pdf",
        "Content Integrity & Citation Standard (Effective Jan 2026)"
    ),
    (
        "Northbridge_Brand_Guidelines.pdf",
        "Brand & Communication Guidelines"
    ),
]


# ==============================================================
# CONFIGURATION
# ==============================================================

# QUERY-LEVEL THRESHOLD:
# The strongest retrieved chunk must reach this score before the
# query is considered sufficiently related to the Northbridge corpus.
MIN_TOP_SCORE = 0.14

# CHUNK-LEVEL THRESHOLD:
# Once the query passes the query-level gate, individual chunks must
# reach this score to be retained as supporting evidence.
MIN_RELEVANCE_SCORE = 0.10

# Maximum number of candidate evidence chunks.
DEFAULT_TOP_K = 4


# Required Northbridge policy disclaimer.
POLICY_DISCLAIMER = (
    "This document reflects Northbridge’s interpretation of currently "
    "available data and community input. Policy conditions may change."
)


# ==============================================================
# DOCUMENT INGESTION & SENTENCE-BASED CHUNKING
# ==============================================================

def load_pdf_chunks(
    path: str,
    doc_name: str,
    max_chars: int = 650
) -> List[Chunk]:
    """
    Extract text from a PDF and create sentence-based chunks.

    Each chunk retains:
    - document name
    - page number
    - text

    This metadata allows retrieved evidence to be cited in the
    final response.
    """

    reader = PdfReader(path)
    chunks: List[Chunk] = []

    for i, page in enumerate(reader.pages):

        page_num = i + 1

        text = (page.extract_text() or "").strip()

        if not text:
            continue

        # Normalize whitespace.
        text = re.sub(r"\s+", " ", text).strip()

        # Split page text into sentences.
        sentences = re.split(
            r"(?<=[.!?])\s+",
            text
        )

        current = ""

        for sentence in sentences:

            sentence = sentence.strip()

            if not sentence:
                continue

            # Continue adding sentences while the chunk remains
            # below the maximum size.
            if len(current) + len(sentence) + 1 <= max_chars:

                current = (
                    current + " " + sentence
                ).strip()

            else:

                if current:

                    chunks.append(
                        Chunk(
                            doc_name=doc_name,
                            page_num=page_num,
                            text=current
                        )
                    )

                current = sentence

        # Save the final chunk for the page.
        if current:

            chunks.append(
                Chunk(
                    doc_name=doc_name,
                    page_num=page_num,
                    text=current
                )
            )

    return chunks


# ==============================================================
# CORPUS CONSTRUCTION
# ==============================================================

def build_corpus() -> List[Chunk]:
    """
    Load all approved Northbridge documents into one searchable corpus.
    """

    corpus: List[Chunk] = []

    for path, name in PDFS:

        if not os.path.exists(path):

            raise FileNotFoundError(
                f"Missing required Northbridge document: {path}"
            )

        document_chunks = load_pdf_chunks(
            path=path,
            doc_name=name
        )

        corpus.extend(document_chunks)

    return corpus


# Build corpus.
CORPUS = build_corpus()


# ==============================================================
# TF-IDF VECTORIZATION
# ==============================================================

VECTORIZER = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)


DOC_MATRIX = VECTORIZER.fit_transform(
    [chunk.text for chunk in CORPUS]
)


# ==============================================================
# RETRIEVAL LAYER
# ==============================================================

def retrieve_top_k(
    query: str,
    k: int = DEFAULT_TOP_K,
    min_score: float = MIN_RELEVANCE_SCORE,
    min_top_score: float = MIN_TOP_SCORE
) -> List[Dict[str, Any]]:
    """
    Retrieve relevant evidence using TF-IDF cosine similarity.

    Two-stage relevance validation:

    Stage 1 — Query-Level Gate
    --------------------------------
    The strongest retrieved result must exceed MIN_TOP_SCORE.

    If the strongest match is too weak, the entire query is treated
    as unsupported and no evidence is returned.

    Stage 2 — Chunk-Level Gate
    --------------------------------
    Once the query passes Stage 1, individual evidence chunks must
    exceed MIN_RELEVANCE_SCORE to be retained.

    This prevents unrelated questions from receiving arbitrary
    citations simply because TF-IDF always ranks something highest.
    """

    query_vector = VECTORIZER.transform(
        [query]
    )

    similarities = cosine_similarity(
        query_vector,
        DOC_MATRIX
    ).flatten()

    top_indices = similarities.argsort()[::-1][:k]

    # No candidates.
    if len(top_indices) == 0:
        return []

    # ----------------------------------------------------------
    # STAGE 1: QUERY-LEVEL RELEVANCE GATE
    # ----------------------------------------------------------

    top_score = float(
        similarities[top_indices[0]]
    )

    if top_score < min_top_score:

        # The corpus does not contain a sufficiently strong match.
        # Return no evidence rather than irrelevant evidence.
        return []

    # ----------------------------------------------------------
    # STAGE 2: CHUNK-LEVEL RELEVANCE GATE
    # ----------------------------------------------------------

    results: List[Dict[str, Any]] = []

    for idx in top_indices:

        score = float(
            similarities[idx]
        )

        # Skip weak individual chunks.
        if score < min_score:
            continue

        chunk = CORPUS[idx]

        # Keep enough text to provide useful context.
        excerpt = chunk.text[:800].strip()

        # Try to end at a clean sentence boundary.
        last_boundary = max(
            excerpt.rfind("."),
            excerpt.rfind("!"),
            excerpt.rfind("?")
        )

        if last_boundary > 120:
            excerpt = excerpt[:last_boundary + 1]

        # Remove unusual leading characters.
        excerpt = re.sub(
            r"^[^A-Za-z0-9]+",
            "",
            excerpt
        )

        results.append({
            "doc_name": chunk.doc_name,
            "page": chunk.page_num,
            "excerpt": excerpt,
            "score": score,
        })

    return results


# ==============================================================
# RETRIEVAL DIAGNOSTIC
# ==============================================================

def retrieval_diagnostic(
    query: str,
    k: int = DEFAULT_TOP_K
) -> List[float]:
    """
    Return the raw top similarity scores before filtering.

    This function is used for evaluation and threshold validation.
    """

    query_vector = VECTORIZER.transform(
        [query]
    )

    similarities = cosine_similarity(
        query_vector,
        DOC_MATRIX
    ).flatten()

    top_indices = similarities.argsort()[::-1][:k]

    return [
        round(float(similarities[idx]), 3)
        for idx in top_indices
    ]


# ==============================================================
# GUARDRAIL POLICY DEFINITIONS
# ==============================================================

# Topics explicitly identified as outside Northbridge's documented
# strategic scope.

OUT_OF_SCOPE_KEYWORDS = [
    "immigration",
    "federal immigration",
    "statewide tax",
    "tax reform",
    "partisan",
    "election",
    "candidate",
    "climate legislation",
    "national tax policy",
]


# Language associated with unsupported causal certainty or prohibited
# impact claims.

PROHIBITED_CLAIM_WORDS = [
    "prove",
    "proves",
    "proven",
    "guarantee",
    "guarantees",
    "guaranteed",
    "eliminate",
    "eliminates",
    "eliminated",
    "directly caused",
    "directly causes",
    "directly cause",
]


# ==============================================================
# RISK CHECK LAYER
# ==============================================================

def risk_check(
    question: str,
    evidence: List[Dict[str, Any]]
) -> Dict[str, Any]:
    """
    Apply deterministic governance checks.

    The system distinguishes between:

    1. Supported questions
    2. Explicitly out-of-scope policy questions
    3. Requests for causal or overclaim language
    4. Questions with insufficient relevant evidence
    """

    q = question.lower()

    # ----------------------------------------------------------
    # OUT-OF-SCOPE CHECK
    # ----------------------------------------------------------

    out_of_scope = any(
        keyword in q
        for keyword in OUT_OF_SCOPE_KEYWORDS
    )

    # ----------------------------------------------------------
    # OVERCLAIM CHECK
    # ----------------------------------------------------------

    asks_for_overclaim = any(
        word in q
        for word in PROHIBITED_CLAIM_WORDS
    )

    # ----------------------------------------------------------
    # EVIDENCE CHECK
    # ----------------------------------------------------------

    # retrieve_top_k() has already applied both relevance gates.
    # Therefore, any evidence remaining here has passed retrieval
    # validation.

    has_support = len(evidence) > 0

    insufficient_evidence = not has_support

    # ----------------------------------------------------------
    # EXPLANATION / AUDIT TRAIL
    # ----------------------------------------------------------

    reasons = []

    if out_of_scope:

        reasons.append(
            "Question touches a topic outside Northbridge's "
            "documented strategic scope."
        )

    if asks_for_overclaim:

        reasons.append(
            "Question requests proof, causality, or certainty that "
            "may exceed documented evidence."
        )

    if insufficient_evidence:

        reasons.append(
            "No sufficiently relevant evidence was found in the "
            "approved Northbridge document corpus."
        )

    return {

        "has_support":
            has_support,

        "insufficient_evidence":
            insufficient_evidence,

        "out_of_scope":
            out_of_scope,

        "asks_for_overclaim":
            asks_for_overclaim,

        "needs_human_review":
            out_of_scope,

        "required_disclaimer":
            POLICY_DISCLAIMER,

        "reasons":
            reasons,
    }


# ==============================================================
# EVIDENCE FORMATTING
# ==============================================================

def format_evidence(
    evidence: List[Dict[str, Any]],
    limit: int = 3
) -> List[str]:
    """
    Convert retrieved chunks into citation-ready evidence lines.
    """

    lines = []

    for item in evidence[:limit]:

        lines.append(
            f'- [{item["doc_name"]}, p.{item["page"]}]'
        )

        lines.append(
            f'  “{item["excerpt"]}”\n'
        )

    return lines


# ==============================================================
# DETERMINISTIC RESPONSE CONSTRUCTION
# ==============================================================

def compose_fallback(
    question: str,
    evidence: List[Dict[str, Any]],
    risk: Dict[str, Any]
) -> str:
    """
    Construct a structured response without requiring an LLM.

    Response behavior depends on the deterministic governance
    classification.
    """

    lines = []

    lines.append(
        f"Question: {question}\n"
    )

    # ==========================================================
    # CASE 1: EXPLICITLY OUTSIDE DOCUMENTED STRATEGIC SCOPE
    # ==========================================================

    if risk["out_of_scope"]:

        lines.append(
            "Answer (Brief):"
        )

        lines.append(
            "- This topic falls outside Northbridge’s currently "
            "documented strategic scope and/or formal policy "
            "position process."
        )

        lines.append(
            "- The assistant will not infer or create an "
            "organizational position that has not been formally "
            "documented."
        )

        lines.append(
            "- Recommended next step: route the question to "
            "Advocacy leadership for governance review.\n"
        )

        # Out-of-scope questions may still have legitimate evidence
        # documenting the organizational scope boundary.

        if evidence:

            lines.append(
                "Evidence (Doc + Page + Excerpt):"
            )

            lines.extend(
                format_evidence(evidence, limit = 2)
            )

        lines.append(
            "Limits:"
        )

        lines.append(
            "- The retrieved materials may document Northbridge’s "
            "scope boundary, but they do not establish a substantive "
            "policy position on this topic.\n"
        )

        lines.append(
            risk["required_disclaimer"]
        )

        return "\n".join(lines)

    # ==========================================================
    # CASE 2: INSUFFICIENT OR IRRELEVANT EVIDENCE
    # ==========================================================

    if risk["insufficient_evidence"]:

        lines.append(
            "Answer (Brief):"
        )

        lines.append(
            "- The provided Northbridge materials do not contain "
            "sufficient documented evidence to answer this question."
        )

        lines.append(
            "- This assistant is limited to approved internal "
            "Northbridge sources.\n"
        )

        lines.append(
            "Evidence:"
        )

        lines.append(
            "- No sufficiently relevant evidence was found in the "
            "approved document set.\n"
        )

        lines.append(
            "Limits:"
        )

        lines.append(
            "- The assistant does not use general knowledge or "
            "speculation when a response cannot be grounded in "
            "approved Northbridge documentation.\n"
        )

        lines.append(
            "Recommended Next Step:"
        )

        lines.append(
            "- Ask a question related to Northbridge’s programs, "
            "advocacy priorities, policies, evaluation findings, "
            "or documented organizational information."
        )

        # Intentionally no policy disclaimer.
        #
        # This is an abstention from answering an unrelated or
        # unsupported question rather than a Northbridge policy
        # briefing.

        return "\n".join(lines)

    # ==========================================================
    # CASE 3: SUPPORTED QUESTION
    # ==========================================================

    combined_evidence = " ".join(
        item["excerpt"]
        for item in evidence[:3]
    ).lower()

    takeaways = []

    # ----------------------------------------------------------
    # ADVOCACY PRIORITIES
    # ----------------------------------------------------------

    if (
        "advocacy priorities" in combined_evidence
        or
        "primary advocacy priorities" in combined_evidence
        or
        "strategic plan priorities" in combined_evidence
    ):

        if "food access" in combined_evidence:

            takeaways.append(
                "Advocacy priorities include improving food "
                "access and nutrition equity."
            )

        if (
            "educational" in combined_evidence
            or
            "youth" in combined_evidence
        ):

            takeaways.append(
                "Advocacy priorities include expanding educational "
                "and youth support programs."
            )

        if "volunteer" in combined_evidence:

            takeaways.append(
                "Advocacy priorities include strengthening "
                "volunteer engagement systems."
            )

        if (
            "data-informed" in combined_evidence
            or
            "data informed" in combined_evidence
        ):

            takeaways.append(
                "Advocacy priorities include advancing "
                "data-informed advocacy rooted in documented "
                "community needs."
            )

    # ----------------------------------------------------------
    # FOOD ACCESS / EASTVIEW
    # ----------------------------------------------------------

    if (
        "limited proximity" in combined_evidence
        or
        "grocery" in combined_evidence
    ):

        takeaways.append(
            "Northbridge documentation identifies food-access "
            "barriers including proximity and transportation "
            "constraints affecting access to grocery resources."
        )

    # ----------------------------------------------------------
    # MULTILINGUAL / ACCESSIBILITY
    # ----------------------------------------------------------

    if "multilingual" in combined_evidence:

        takeaways.append(
            "Community feedback highlights the importance of "
            "multilingual communication and culturally appropriate "
            "service approaches."
        )

    # ----------------------------------------------------------
    # FOOD-INSECURITY CAUSAL LIMIT
    # ----------------------------------------------------------

    if (
        "does not assert" in combined_evidence
        and
        "eliminate food insecurity" in combined_evidence
    ):

        takeaways.append(
            "Northbridge explicitly avoids claiming that a single "
            "policy or program intervention eliminates food "
            "insecurity."
        )

    # ----------------------------------------------------------
    # BUILD ANSWER
    # ----------------------------------------------------------

    lines.append(
        "Answer (Brief):"
    )

    if takeaways:

        # Remove duplicate takeaways while preserving order.
        unique_takeaways = list(
            dict.fromkeys(takeaways)
        )

        for takeaway in unique_takeaways[:4]:

            lines.append(
                f"- {takeaway}"
            )

        lines.append("")

    else:

        lines.append(
            "- The approved Northbridge documents contain relevant "
            "information on this topic. See the cited evidence below.\n"
        )

    # ----------------------------------------------------------
    # EVIDENCE
    # ----------------------------------------------------------

    lines.append(
        "Evidence (Doc + Page + Excerpt):"
    )

    lines.extend(
        format_evidence(evidence)
    )

    # ----------------------------------------------------------
    # OVERCLAIM / CAUSALITY LIMIT
    # ----------------------------------------------------------

    if risk["asks_for_overclaim"]:

        lines.append(
            "Limits / What the Documentation Does Not Establish:"
        )

        lines.append(
            "- The available evidence does not independently prove "
            "elimination, guarantee outcomes, or establish causality "
            "unless supported by formal evaluation methods.\n"
        )

    # ----------------------------------------------------------
    # DISCLAIMER
    # ----------------------------------------------------------

    lines.append(
        risk["required_disclaimer"]
    )

    return "\n".join(lines)


# ==============================================================
# LLM SYSTEM PROMPT
# ==============================================================

SYSTEM_PROMPT = f"""
You are the Northbridge Policy Briefing Assistant for the
Advocacy & Community Issues team.

You operate only on approved Northbridge evidence supplied to you.

NON-NEGOTIABLE RULES:

1. Use ONLY the provided evidence excerpts to support factual
   claims, statistics, organizational positions, or recommendations.

2. Never use outside knowledge to fill gaps in the retrieved evidence.

3. Every factual or numerical claim must be traceable to the supplied
   evidence and should use this citation format:

   [Document Name, p.#] — "short supporting excerpt"

4. If the guardrail indicates insufficient evidence:
   - Clearly state that the approved Northbridge materials do not
     contain sufficient documented evidence.
   - Do not answer the question using general knowledge.
   - Do not present irrelevant evidence.
   - Do not speculate.

5. If the guardrail indicates that the question is outside
   Northbridge's documented strategic scope:
   - Clearly state that it is outside documented scope.
   - Do not create or infer a Northbridge policy position.
   - Recommend Advocacy leadership review.
   - Evidence may be used only to document the scope boundary.

6. If the question requests proof, guarantees, elimination, or
   causal certainty:
   - Avoid unsupported causal language.
   - Clearly explain what the documentation does and does not establish.

7. Use a respectful, clear, concise tone appropriate for policy
   stakeholders.

8. For supported policy or advocacy responses, include this disclaimer:

   "{POLICY_DISCLAIMER}"

9. For completely unrelated questions with insufficient evidence:
   - Abstain clearly.
   - Do not attach unrelated citations merely to provide an
     Evidence section.

OUTPUT FORMAT FOR SUPPORTED QUESTIONS:

Answer (Brief):
- 2–6 concise bullets

Evidence (Doc + Page + Excerpt):
- cited evidence

Limits:
- include when necessary

Disclaimer:
- required policy disclaimer

OUTPUT FORMAT FOR INSUFFICIENT-EVIDENCE QUESTIONS:

Answer (Brief):
- State that the approved materials do not contain sufficient
  documented evidence.
- State that the assistant is limited to approved Northbridge sources.

Evidence:
- State that no sufficiently relevant evidence was found.

Limits:
- State that general knowledge or speculation will not be used.

Recommended Next Step:
- Suggest asking a Northbridge-related question.
""".strip()


# ==============================================================
# MAIN ANSWER FUNCTION
# ==============================================================

def answer(
    question: str,
    top_k: int = DEFAULT_TOP_K
) -> str:
    """
    Main orchestration pipeline.

    Question
        ↓
    TF-IDF Retrieval
        ↓
    Query-Level Relevance Gate
        ↓
    Chunk-Level Relevance Gate
        ↓
    Deterministic Guardrails
        ↓
    Structured Response
    """

    question = question.strip()

    if not question:

        return (
            "Please enter a question."
        )

    # ----------------------------------------------------------
    # RETRIEVAL
    # ----------------------------------------------------------

    evidence = retrieve_top_k(
        question,
        k=top_k
    )

    # ----------------------------------------------------------
    # GUARDRAILS
    # ----------------------------------------------------------

    risk = risk_check(
        question,
        evidence
    )

    # ----------------------------------------------------------
    # DETERMINISTIC GOVERNANCE DECISION
    # ----------------------------------------------------------
    #
    # Unsupported and explicitly out-of-scope questions are handled
    # deterministically even if OpenAI is enabled.
    #
    # This prevents an LLM from overriding the governance layer.

    if (
        risk["insufficient_evidence"]
        or
        risk["out_of_scope"]
    ):

        return compose_fallback(
            question,
            evidence,
            risk
        )

    # ----------------------------------------------------------
    # DETERMINISTIC MODE
    # ----------------------------------------------------------

    if not USE_OPENAI:

        return compose_fallback(
            question,
            evidence,
            risk
        )

    # ----------------------------------------------------------
    # OPTIONAL LLM MODE
    # ----------------------------------------------------------

    response = client.chat.completions.create(

        model="gpt-4.1-mini",

        messages=[

            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content": (
                    f"Question:\n{question}\n\n"
                    f"Retrieved evidence:\n"
                    f"{json.dumps(evidence, indent=2)}\n\n"
                    f"Guardrail assessment:\n"
                    f"{json.dumps(risk, indent=2)}"
                )
            },
        ],

        temperature=0.2
    )

    return response.choices[0].message.content


# ==============================================================
# EVALUATION / ADVERSARIAL TEST SET
# ==============================================================

EVALUATION_QUESTIONS = [

    # ----------------------------------------------------------
    # SUPPORTED
    # ----------------------------------------------------------

    {
        "category": "Supported",
        "question":
            "What are Northbridge's main advocacy priorities?"
    },

    {
        "category": "Supported",
        "question":
            "What evidence does Northbridge have regarding food access in Eastview?"
    },

    {
        "category": "Supported",
        "question":
            "What is Northbridge's position on volunteer certification standards?"
    },

    # ----------------------------------------------------------
    # OVERCLAIM / CAUSALITY
    # ----------------------------------------------------------

    {
        "category": "Overclaim",
        "question":
            "Has Northbridge proven that its food programs eliminate food insecurity?"
    },

    {
        "category": "Overclaim",
        "question":
            "Does Northbridge tutoring directly cause improved academic performance?"
    },

    # ----------------------------------------------------------
    # EXPLICITLY OUT OF SCOPE
    # ----------------------------------------------------------

    {
        "category": "Out of Scope",
        "question":
            "What is Northbridge's position on federal immigration policy?"
    },

    {
        "category": "Out of Scope",
        "question":
            "Does Northbridge support statewide tax reform?"
    },

    {
        "category": "Out of Scope",
        "question":
            "What is Northbridge's position on climate legislation?"
    },

    # ----------------------------------------------------------
    # UNRELATED / INSUFFICIENT EVIDENCE
    # ----------------------------------------------------------

    {
        "category": "Insufficient Evidence",
        "question":
            "How can I make pasta?"
    },

    {
        "category": "Insufficient Evidence",
        "question":
            "Show me the recipe."
    },

    {
        "category": "Insufficient Evidence",
        "question":
            "Who won the Super Bowl?"
    },

    {
        "category": "Insufficient Evidence",
        "question":
            "What laptop should I buy?"
    },
]


# ==============================================================
# EVALUATION FUNCTION
# ==============================================================

def run_evaluation():
    """
    Run supported, overclaim, out-of-scope, and adversarial tests.

    Displays:
    - expected category
    - question
    - raw similarity scores
    - query-level relevance gate result
    - evidence retained
    - guardrail classification

    This allows the retrieval thresholds to be evaluated
    transparently.
    """

    print("\n")

    print(
        "=" * 90
    )

    print(
        "NORTHBRIDGE POLICY BRIEFING ASSISTANT — EVALUATION"
    )

    print(
        "=" * 90
    )

    for test in EVALUATION_QUESTIONS:

        question = test["question"]
        category = test["category"]

        # Raw scores before filtering.
        raw_scores = retrieval_diagnostic(
            question
        )

        # Final evidence after both gates.
        evidence = retrieve_top_k(
            question
        )

        # Guardrail assessment.
        risk = risk_check(
            question,
            evidence
        )

        print(
            "\n" + "-" * 90
        )

        print(
            f"Expected Category: {category}"
        )

        print(
            f"Question: {question}"
        )

        print(
            f"Top Raw Similarity Scores: {raw_scores}"
        )

        # ------------------------------------------------------
        # QUERY-LEVEL GATE RESULT
        # ------------------------------------------------------

        if raw_scores:

            query_gate = (
                "PASS"
                if raw_scores[0] >= MIN_TOP_SCORE
                else
                "REJECT"
            )

            print(
                f"Query-Level Gate: {query_gate} "
                f"(threshold = {MIN_TOP_SCORE})"
            )

        else:

            print(
                "Query-Level Gate: REJECT "
                "(no retrieval candidates)"
            )

        # ------------------------------------------------------
        # RETRIEVAL RESULTS
        # ------------------------------------------------------

        print(
            f"Evidence Retained After Threshold: "
            f"{len(evidence)}"
        )

        # ------------------------------------------------------
        # GUARDRAIL RESULTS
        # ------------------------------------------------------

        print(
            f"Has Support: "
            f"{risk['has_support']}"
        )

        print(
            f"Out of Scope: "
            f"{risk['out_of_scope']}"
        )

        print(
            f"Overclaim Risk: "
            f"{risk['asks_for_overclaim']}"
        )

        print(
            f"Insufficient Evidence: "
            f"{risk['insufficient_evidence']}"
        )

        # ------------------------------------------------------
        # OPTIONAL REASONS
        # ------------------------------------------------------

        if risk["reasons"]:

            print(
                "Guardrail Reasons:"
            )

            for reason in risk["reasons"]:

                print(
                    f"  - {reason}"
                )

    print(
        "\n" + "=" * 90
    )


# ==============================================================
# INTERACTIVE DEMO
# ==============================================================

def run_assistant():
    """
    Run the Northbridge assistant interactively.
    """

    print(
        "\nNorthbridge Policy Briefing Assistant"
    )

    print(
        "Type a question (or 'exit').\n"
    )

    while True:

        question = input(
            "Question: "
        ).strip()

        # ------------------------------------------------------
        # EXIT
        # ------------------------------------------------------

        if question.lower() in {
            "exit",
            "quit",
            "q"
        }:

            print(
                "\nExiting Northbridge Policy Briefing Assistant."
            )

            break

        # ------------------------------------------------------
        # EMPTY QUESTION
        # ------------------------------------------------------

        if not question:

            print(
                "\nPlease enter a question.\n"
            )

            continue

        # ------------------------------------------------------
        # ANSWER
        # ------------------------------------------------------

        print()

        print(
            answer(question)
        )

        print(
            "\n" + "-" * 80 + "\n"
        )

## Evaluation & Guardrail Testing

The assistant is evaluated across four types of scenarios:

1. **Supported queries** — questions directly addressed by approved Northbridge documents.
2. **Overclaim / causality queries** — questions that request proof or causal conclusions beyond the available evidence.
3. **Out-of-scope policy queries** — topics explicitly outside Northbridge's documented strategic scope.
4. **Insufficient-evidence queries** — unrelated questions that should trigger abstention rather than unsupported responses.

Retrieval uses a two-stage relevance gate:
- **Query-level threshold:** 0.14
- **Evidence chunk threshold:** 0.10

The evaluation checks whether the assistant retrieves relevant evidence, activates the appropriate guardrails, and abstains when sufficient evidence is unavailable.

In [15]:
run_evaluation()



NORTHBRIDGE POLICY BRIEFING ASSISTANT — EVALUATION

------------------------------------------------------------------------------------------
Expected Category: Supported
Question: What are Northbridge's main advocacy priorities?
Top Raw Similarity Scores: [0.26, 0.167, 0.09, 0.074]
Query-Level Gate: PASS (threshold = 0.14)
Evidence Retained After Threshold: 2
Has Support: True
Out of Scope: False
Overclaim Risk: False
Insufficient Evidence: False

------------------------------------------------------------------------------------------
Expected Category: Supported
Question: What evidence does Northbridge have regarding food access in Eastview?
Top Raw Similarity Scores: [0.353, 0.121, 0.12, 0.113]
Query-Level Gate: PASS (threshold = 0.14)
Evidence Retained After Threshold: 4
Has Support: True
Out of Scope: False
Overclaim Risk: False
Insufficient Evidence: False

------------------------------------------------------------------------------------------
Expected Category: Supported

## Interactive Policy Briefing Interface

The assistant accepts user questions and returns:
- A concise, evidence-grounded response
- Document name + page citations
- Relevant evidence excerpts
- Guardrail-based limits for unsupported causal claims
- Scope escalation when a topic falls outside documented priorities
- Abstention when sufficiently relevant evidence is unavailable
- Required policy disclaimer when applicable

In [16]:
run_assistant()


Northbridge Policy Briefing Assistant
Type a question (or 'exit').


Question: What are Northbridge's main advocacy priorities?

Answer (Brief):
- Advocacy priorities include improving food access and nutrition equity.
- Advocacy priorities include expanding educational and youth support programs.
- Advocacy priorities include strengthening volunteer engagement systems.
- Advocacy priorities include advancing data-informed advocacy rooted in documented community needs.

Evidence (Doc + Page + Excerpt):
- [Advocacy & Community Issues FAQ (2026), p.1]
  “Northbridge Community Alliance Advocacy & Community Issues FAQ Internal Reference Guide – 2026 1. What are Northbridge’s primary advocacy priorities? Northbridge’s advocacy efforts align with the 2025–2028 Strategic Plan priorities. These include improving food access and nutrition equity, expanding educational and youth support programs, strengthening volunteer engagement systems, and advancing data-informed advocacy rooted in document